# Entrenamiento Mechatronics AI en Colab

Entorno de ejecución → Cambiar tipo → **GPU T4**.

En Drive debe existir `MyDrive/mechatronics-ai/dataset/` con `data.yaml` y las carpetas `train/ val/ test/` generadas por `scripts/split_dataset.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mechatronics-ai'

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## Entrenar (transfer learning desde YOLO11n)

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11n.pt')
modelo.train(data=f'{BASE}/dataset/data.yaml', epochs=80, imgsz=640, batch=32,
             patience=20, device=0, project=f'{BASE}/runs', name='mechatronics', exist_ok=True)

## Evaluar en test

In [ ]:
best = YOLO(f'{BASE}/runs/mechatronics/weights/best.pt')
res = best.val(data=f'{BASE}/dataset/data.yaml', split='test')
print('P', res.box.mp, 'R', res.box.mr, 'mAP50', res.box.map50, 'mAP50-95', res.box.map)

## Exportar

Descarga `best.pt` a `models/` en tu PC y exporta allí a OpenVINO con `python scripts/export.py --model models/best.pt` (la exportación OpenVINO conviene hacerla en la CPU destino).

In [ ]:
from google.colab import files
files.download(f'{BASE}/runs/mechatronics/weights/best.pt')